# Softmax·Temperature·Top-k/p 샘플링 실습 - 추가 실습 코드: Temperature가 분포에 미치는 영향

- Tutorial ID: `mod-softmax-temperature-lab`
- Tutorial: Softmax·Temperature·Top-k/p 샘플링 실습
- Section ID: `practice-temperature-effect`
- Section: 추가 실습 코드: Temperature가 분포에 미치는 영향


In [ ]:
# ============================================================
# 코드 읽는 법 — 추가 실습 코드: Temperature가 분포에 미치는 영향
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

def softmax_with_temperature(logits, temperature=1.0):
    """Temperature가 적용된 Softmax"""
    scaled = logits / temperature
    exp_scaled = np.exp(scaled - np.max(scaled))
    return exp_scaled / np.sum(exp_scaled)

# 예시 로짓 (모델의 원시 출력)
logits = np.array([2.0, 1.5, 0.5, 0.3, 0.1])
tokens = ["좋다", "좋아요", "괜찮다", "그렇다", "싫다"]

print("=" * 50)
print("Temperature에 따른 확률 분포 변화")
print("=" * 50)
print(f"\\n원본 로짓: {logits}")
print(f"토큰: {tokens}")

temperatures = [0.1, 0.5, 1.0, 1.5, 2.0]

for T in temperatures:
        probs = softmax_with_temperature(logits, T)
    bar = lambda p: "█" * int(p * 30)
    
    print(f"\\n━━━ Temperature = {T} ━━━")
        for token, prob in zip(tokens, probs):
        print(f"  {token:5s} {prob:.3f} {bar(prob)}")
    
    # 엔트로피 계산 (분포의 불확실성)
    entropy = -np.sum(probs * np.log(probs + 1e-10))
    print(f"  엔트로피: {entropy:.3f}")

print("\\n" + "=" * 50)
print("📊 분석:")
print("  T → 0: Argmax에 가까움 (결정론적)")
print("  T = 1: 원래 분포")
print("  T → ∞: 균등분포에 가까움 (랜덤)")